## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("hf_token_rp")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [2]:
import os
os.environ.setdefault("USE_TF", "0")
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILE0 = "Phase_1_Pretraining/Train/train_supreme_court.json"
TRAIN_FILE1 = "Phase_1_Pretraining/Train/train_high_courts.json"
TRAIN_FILE2 = "Phase_1_Pretraining/Train/train_acts.json"
TRAIN_FILE3 = "Phase_1_Pretraining/Train/train_tribunals.json"

TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TRAIN_TOKENS = 200_000_000
VAL_TOKENS = 5_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 3
SEED = 42
random.seed(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 640
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95

# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-encdec-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

# ------------------------------------------------------------
# Run-mode switches
# ------------------------------------------------------------
# False: load the saved BPE tokenizer used for every submission (recommended).
# True : retrain the BPE tokenizer from the corpus (slow, pure Python).
TRAIN_NEW_TOKENIZER = False
TOKENIZER_INPUT_PATH = "/kaggle/input/datasets/sg123321/tokenizerdat/legal_bpe.json"

# Optional pre-tokenized corpus (train/val token arrays). If it is not attached,
# the mixed corpus is built and tokenized in this notebook and cached in OUTPUT_DIR.
TOKENIZED_CACHE_INPUT = "/kaggle/input/datasets/sg123321/tokenizerdat/tokenized_data.npz"

# Full-corpus token statistics (used for the report). Very slow; off by default.
COUNT_CORPUS_TOKENS = False

# Resume an interrupted run from a Trainer checkpoint directory (None = start from random init).
RESUME_FROM_CHECKPOINT = None


Configuration loaded.


## 4. Download Training Data From Hugging Face

In [3]:
from huggingface_hub import hf_hub_download

# Download all four training files
TRAIN_FILES = {
    "supreme_court": hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=TRAIN_FILE0,
        repo_type="dataset"
    ),
    "high_courts": hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=TRAIN_FILE1,
        repo_type="dataset"
    ),
    "acts": hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=TRAIN_FILE2,
        repo_type="dataset"
    ),
    "tribunals": hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=TRAIN_FILE3,
        repo_type="dataset"
    )
}

for source, path in TRAIN_FILES.items():
    print(f"{source}: {path}")

Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

supreme_court: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json
high_courts: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json
acts: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json
tribunals: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json


## 5. Load Tokenizer

In [3]:

# tokenizer.pad_token = tokenizer.eos_token
# print("Vocabulary size:", len(tokenizer))

#BPE tokenizer

def train_tokenizer(train_text, vocab_size):
    tokenizer = ByteLevelBPE(vocab_size=vocab_size)
    tokenizer.train([train_text])
    return tokenizer

# ============================================================
# QUESTION 5 — OPTIMIZED BYTE-LEVEL BPE
# ============================================================

import json
from collections import Counter, defaultdict
from tqdm.auto import tqdm


BPE_SPECIALS = [
    "<pad>",
    "<unk>",
    "<s>",
    "</s>"
]

# End-of-word marker.
# UTF-8 bytes are integers 0..255, so 256 cannot collide.
EOW = 256


class ByteLevelBPE:
    """
    Deterministic byte-level BPE tokenizer.

    - Initial units are UTF-8 bytes.
    - EOW marks the end of each word.
    - Case is preserved.
    - BPE never crosses word boundaries.
    - Ties are broken lexicographically.
    - Four special tokens are fixed at the beginning.
    """

    def __init__(self, vocab_size=5000):

        self.vocab_size = vocab_size
        self.merge_ranks = {}
        self.specials = list(BPE_SPECIALS)

        # List of learned merges.
        #
        # Each element is:
        #     (left_symbol, right_symbol)
        #
        self.merges = []

        # Vocabulary.
        self.itos = []
        self.stoi = {}

        # Internal merge rules.
        #
        # (left, right) -> merged_symbol
        #
        self.merge_rules = {}

        # Final representations of words seen during training.
        self.word_cache = {}
        self.token_to_symbol = {}

    # --------------------------------------------------------
    # Convert a byte/symbol tuple into a deterministic string
    # --------------------------------------------------------
    @staticmethod
    def string_to_symbol(token):
        # Standalone end-of-word marker
        if token == "</w>":
            return (EOW,)
    
        # Merged symbol ending with EOW
        if token.endswith("</w>>"):
            hex_part = token[3:-5]
            byte_part = tuple(
                int(hex_part[i:i+2], 16)
                for i in range(0, len(hex_part), 2)
            )
            return byte_part + (EOW,)
    
        # Ordinary byte or merged byte symbol
        hex_part = token[3:-1]
        return tuple(
            int(hex_part[i:i+2], 16)
            for i in range(0, len(hex_part), 2)
        )
    @staticmethod
    def symbol_to_string(symbol):

        # EOW is always the last symbol when present.
        if symbol and symbol[-1] == EOW:

            byte_part = symbol[:-1]

            if len(byte_part) == 0:
                return "</w>"

            hex_part = "".join(
                f"{b:02x}"
                for b in byte_part
            )

            return f"<0x{hex_part}</w>>"

        hex_part = "".join(
            f"{b:02x}"
            for b in symbol
        )

        return f"<0x{hex_part}>"
    def _build_merge_ranks(self):
        self.merge_ranks = {
            pair: rank
            for rank, pair in enumerate(self.merge_rules)
        }
    # --------------------------------------------------------
    # Deterministic ordering key for symbols
    # --------------------------------------------------------

    @staticmethod
    def symbol_key(symbol):
        return tuple(symbol)

    # --------------------------------------------------------
    # Convert a word into initial UTF-8 byte symbols
    # --------------------------------------------------------

    @staticmethod
    def word_to_symbols(word):

        byte_values = list(
            word.encode("utf-8")
        )

        symbols = [
            (b,)
            for b in byte_values
        ]

        # Explicit word boundary.
        symbols.append((EOW,))

        return symbols

    # --------------------------------------------------------
    # Count all pairs once initially
    # --------------------------------------------------------

    def _initial_pair_counts(
        self,
        word_symbols,
        word_counts
    ):

        pair_counts = Counter()

        # pair_words tells us which words contain a pair.
        pair_words = defaultdict(set)

        for word, symbols in word_symbols.items():

            freq = word_counts[word]

            for i in range(len(symbols) - 1):

                pair = (
                    symbols[i],
                    symbols[i + 1]
                )

                pair_counts[pair] += freq
                pair_words[pair].add(word)

        return pair_counts, pair_words

    # --------------------------------------------------------
    # Merge one pair in one word
    # --------------------------------------------------------

    @staticmethod
    def _merge_word(symbols, pair):

        left, right = pair

        new_symbol = left + right

        result = []

        i = 0

        while i < len(symbols):

            if (
                i + 1 < len(symbols)
                and symbols[i] == left
                and symbols[i + 1] == right
            ):

                result.append(new_symbol)

                i += 2

            else:

                result.append(symbols[i])

                i += 1

        return result

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    def train(self, texts):

        print("=" * 70)
        print("Training BYTE-LEVEL BPE")
        print("=" * 70)

        # ----------------------------------------------------
        # Count unique words.
        # ----------------------------------------------------

        word_counts = Counter()

        for text in texts:
            if not isinstance(text, str):
                continue

            for word in text.split():

                word_counts[word] += 1

        print(
            f"Unique words: {len(word_counts):,}"
        )

        # ----------------------------------------------------
        # Initial representation of each word.
        # ----------------------------------------------------

        word_symbols = {
            word: self.word_to_symbols(word)
            for word in word_counts
        }

        # ----------------------------------------------------
        # Initial vocabulary symbols.
        # ----------------------------------------------------

        initial_symbols = [
            (b,)
            for b in range(256)
        ]

        initial_symbols.append((EOW,))

        # Four specials + initial byte symbols.
        current_vocab_size = (
            len(BPE_SPECIALS)
            + len(initial_symbols)
        )

        print(
            f"Initial vocabulary size: "
            f"{current_vocab_size:,}"
        )

        if current_vocab_size > self.vocab_size:

            raise ValueError(
                "Initial byte vocabulary already exceeds "
                "the requested vocabulary size."
            )

        # ----------------------------------------------------
        # Count pairs once.
        # ----------------------------------------------------

        pair_counts, pair_words = \
            self._initial_pair_counts(
                word_symbols,
                word_counts
            )

        # ----------------------------------------------------
        # Main BPE loop.
        # ----------------------------------------------------

        merge_no = 0

        while current_vocab_size < self.vocab_size:

            if not pair_counts:
                break

            # Ignore pairs occurring only once.
            valid_pairs = [
                pair
                for pair, count in pair_counts.items()
                if count > 1
            ]

            if not valid_pairs:
                print(
                    "No pair occurs more than once. "
                    "Stopping early."
                )
                break

            # Highest frequency first.
            # Lexicographically smallest pair on ties.
            best_pair = min(
                valid_pairs,
                key=lambda pair: (
                    -pair_counts[pair],
                    self.symbol_key(pair[0]),
                    self.symbol_key(pair[1])
                )
            )

            best_count = pair_counts[best_pair]

            left, right = best_pair
            new_symbol = left + right

            # ------------------------------------------------
            # Record merge.
            # ------------------------------------------------

            self.merge_rules[best_pair] = new_symbol

            self.merges.append(
                (
                    self.symbol_to_string(left),
                    self.symbol_to_string(right)
                )
            )

            # ------------------------------------------------
            # Words affected by this merge.
            # ------------------------------------------------

            affected_words = list(
                pair_words.get(best_pair, set())
            )

            # Remove the selected pair's old contribution.
            del pair_counts[best_pair]

            pair_words.pop(
                best_pair,
                None
            )

            # ------------------------------------------------
            # Update each affected word.
            # ------------------------------------------------

            for word in affected_words:

                old_symbols = word_symbols[word]

                old_pairs = []

                for i in range(
                    len(old_symbols) - 1
                ):

                    old_pairs.append(
                        (
                            old_symbols[i],
                            old_symbols[i + 1]
                        )
                    )

                # Remove old pair contributions.
                freq = word_counts[word]

                for pair in old_pairs:

                    pair_counts[pair] -= freq

                    if pair_counts[pair] <= 0:
                        pair_counts.pop(
                            pair,
                            None
                        )

                    if pair in pair_words:
                        pair_words[pair].discard(word)

                        if not pair_words[pair]:
                            pair_words.pop(
                                pair,
                                None
                            )

                # Perform merge.
                new_symbols = self._merge_word(
                    old_symbols,
                    best_pair
                )

                word_symbols[word] = new_symbols

                # Add new pair contributions.
                new_pairs = []

                for i in range(
                    len(new_symbols) - 1
                ):

                    new_pairs.append(
                        (
                            new_symbols[i],
                            new_symbols[i + 1]
                        )
                    )

                for pair in new_pairs:

                    pair_counts[pair] += freq
                    pair_words[pair].add(word)

            current_vocab_size += 1
            merge_no += 1

            if merge_no % 250 == 0:

                print(
                    f"Merges: {merge_no:4d} | "
                    f"Vocab: {current_vocab_size:5d} | "
                    f"Best pair frequency: {best_count:,}"
                )

        # ----------------------------------------------------
        # Construct final vocabulary.
        #
        # First:
        #   <pad>, <unk>, <s>, </s>
        #
        # Then initial symbols.
        #
        # Then symbols introduced by merges in merge order.
        # ----------------------------------------------------

        merged_symbols = list(
            self.merge_rules.values()
        )

        self.itos = (
            list(BPE_SPECIALS)
            +
            [
                self.symbol_to_string(s)
                for s in initial_symbols
            ]
            +
            [
                self.symbol_to_string(s)
                for s in merged_symbols
            ]
        )

        self.token_to_symbol = {}

        for symbol in initial_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        for symbol in merged_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        # Remove duplicates while preserving order.
        final_itos = []
        seen = set()

        for token in self.itos:

            if token not in seen:

                seen.add(token)
                final_itos.append(token)

        self.itos = final_itos[
            :self.vocab_size
        ]

        self.stoi = {
            token: i
            for i, token in enumerate(self.itos)
        }

        # Save representations of training words.
        self.word_cache = word_symbols

        print(
            "\nTraining complete."
        )

        print(
            f"Final vocabulary size: "
            f"{len(self.itos):,}"
        )

        print(
            f"Number of merges: "
            f"{len(self.merges):,}"
        )
        self._build_merge_ranks()
        return self

    # --------------------------------------------------------
    # Tokenize a word
    # --------------------------------------------------------

    def tokenize_word(self, word):
        # Use cached representation for training words.
        if word in self.word_cache:
            symbols = self.word_cache[word]
        else:
            symbols = self.word_to_symbols(word)
    
            while len(symbols) > 1:
                # Find adjacent pairs that have learned merges.
                candidates = [
                    (self.merge_ranks[(symbols[i], symbols[i + 1])], i)
                    for i in range(len(symbols) - 1)
                    if (symbols[i], symbols[i + 1]) in self.merge_ranks
                ]
    
                if not candidates:
                    break
    
                # Apply the earliest learned merge.
                _, best_idx = min(candidates)
    
                pair = (symbols[best_idx], symbols[best_idx + 1])
                new_symbol = self.merge_rules[pair]
    
                # Merge all non-overlapping occurrences.
                symbols = self._merge_word(symbols, pair)
    
        return [
            self.symbol_to_string(symbol)
            for symbol in symbols
        ]

    # --------------------------------------------------------
    # Tokenize complete sentence
    # --------------------------------------------------------

    def tokenize(self, text):

        tokens = []

        for word in text.split():

            tokens.extend(
                self.tokenize_word(word)
            )

        return tokens

    def encode(self, text):
        tokens = self.tokenize(text)

        ids = []
        for token in tokens:
            if token in self.stoi:
                ids.append(self.stoi[token])
            else:
                # unknown token
                ids.append(self.stoi["<unk>"])

        return ids

    def decode(self, ids):
      byte_values = []

      for idx in ids:
          token = self.itos[idx]

          if token in BPE_SPECIALS:
              continue

          symbol = self.token_to_symbol[token]

          for b in symbol:
              if b == EOW:
                  byte_values.append(ord(" "))
              else:
                  byte_values.append(b)

      # Remove the final space
      if byte_values and byte_values[-1] == ord(" "):
          byte_values.pop()

      return bytes(byte_values).decode("utf-8")

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = data["merges"]
        tokenizer.itos = data["itos"]
        tokenizer.stoi = {
            token: i for i, token in enumerate(tokenizer.itos)
        }
    
        # Initial symbols
        initial_symbols = [(b,) for b in range(256)]
        initial_symbols.append((EOW,))
    
        tokenizer.token_to_symbol = {
            tokenizer.symbol_to_string(symbol): symbol
            for symbol in initial_symbols
        }
    
        # Reconstruct merge rules in training order
        for left_str, right_str in tokenizer.merges:
            left = tokenizer.string_to_symbol(left_str)
            right = tokenizer.string_to_symbol(right_str)
            merged = left + right
    
            tokenizer.merge_rules[(left, right)] = merged
            merged_str = tokenizer.symbol_to_string(merged)
            tokenizer.token_to_symbol[merged_str] = merged
        tokenizer._build_merge_ranks()  
        return tokenizer

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    def save(self, path):

        data = {
            "merges": self.merges,
            "itos": self.itos,
            "vocab_size": self.vocab_size
        }

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                ensure_ascii=False
            )

        print(
            f"Saved: {path}"
        )

# if __name__ == "__main__":
#     train_text = """
#     Harry went to Hogwarts.
#     Harry met Ron and Hermione.
#     Hogwarts was a magical school.
#     """

#     tokenizer = train_tokenizer(train_text, 5000)

#     print("Vocab size:", tokenizer.vocab_size)

#     text = "Harry went to Hogwarts."

#     ids = tokenizer.encode(text)
#     decoded = tokenizer.decode(ids)

#     print("Text:", text)
#     print("IDs:", ids)
#     print("Decoded:", decoded)
#     print("Match:", text == decoded)


In [6]:
!pip install ijson
import os
import ijson

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

VOCAB_SIZE = 16000
MAX_CHARS_PER_SOURCE = 5_000_000

TOKENIZER_PATH = os.path.join(
    OUTPUT_DIR, "legal_bpe.json"
)


# ------------------------------------------------------------
# Extract text from a record
# ------------------------------------------------------------

def get_text(record):
    if isinstance(record, str):
        return record

    if isinstance(record, dict):
        for key in [
            "text", "content", "judgment",
            "document", "full_text", "body"
        ]:
            if isinstance(record.get(key), str):
                return record[key]

    raise ValueError(
        f"Unable to extract text. Record type: {type(record)}"
    )


# ------------------------------------------------------------
# Stream text from JSON files
# ------------------------------------------------------------

def iter_texts(file_path):
    with open(file_path, "rb") as f:
        for record in ijson.items(f, "item"):
            text = get_text(record)
            if text.strip():
                yield text


# ------------------------------------------------------------
# Create bounded training sample
# ------------------------------------------------------------

def all_documents():
    for source, path in TRAIN_FILES.items():
        print(f"Processing {source}...")

        total_chars = 0

        for text in iter_texts(path):
            remaining = MAX_CHARS_PER_SOURCE - total_chars

            if remaining <= 0:
                break

            text = text[:remaining]

            if text.strip():
                yield text
                total_chars += len(text)

        print(f"{source}: {total_chars:,} characters")


# ------------------------------------------------------------
# Train and save tokenizer
# ------------------------------------------------------------

if TRAIN_NEW_TOKENIZER:
    tokenizer = ByteLevelBPE(vocab_size=VOCAB_SIZE)
    tokenizer.train(all_documents())

    tokenizer.save(TOKENIZER_PATH)

    print(f"Tokenizer saved at: {TOKENIZER_PATH}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 4.5 MB/s eta 0:00:00
Training BYTE-LEVEL BPE
Processing supreme_court...
supreme_court: 5,000,000 characters
Processing high_courts...
high_courts: 5,000,000 characters
Processing acts...
acts: 5,000,000 characters
Processing tribunals...
tribunals: 5,000,000 characters
Unique words: 112,431
Initial vocabulary size: 261
Merges:  250 | Vocab:   511 | Best pair frequency: 8,421


KeyboardInterrupt: 

In [4]:
# Load the tokenizer: the one trained above, or the saved tokenizer used for the submissions.
tokenizer = ByteLevelBPE.load(
    TOKENIZER_PATH if TRAIN_NEW_TOKENIZER else TOKENIZER_INPUT_PATH
)

print("Vocabulary size:", len(tokenizer.itos))
print("Number of merges:", len(tokenizer.merges))

# Special-token ids (needed by the corpus tokenization step below).
PAD_ID = tokenizer.stoi["<pad>"]   # 0
UNK_ID = tokenizer.stoi["<unk>"]   # 1
BOS_ID = tokenizer.stoi["<s>"]     # 2
EOS_ID = tokenizer.stoi["</s>"]    # 3


Vocabulary size: 16000
Number of merges: 15739


In [27]:
import json
from tqdm.auto import tqdm
from pathlib import Path

# ------------------------------------------------------------
# Count tokens in one source
# ------------------------------------------------------------


def count_source_tokens(tokenizer, file_path):
    total_tokens = 0
    total_docs = 0
    total_chars = 0

    for text in tqdm(
        iter_texts(file_path),
        desc=Path(file_path).name
    ):
        ids = tokenizer.encode(text)

        total_tokens += len(ids)
        total_docs += 1
        total_chars += len(text)

        del ids

    return {
        "documents": total_docs,
        "characters": total_chars,
        "tokens": total_tokens,
    }


# ------------------------------------------------------------
# Count tokens in all sources
# ------------------------------------------------------------

def count_all_sources(tokenizer):
    results = {}

    for source, file_path in TRAIN_FILES.items():
        print(f"\nCounting {source}...")
        results[source] = count_source_tokens(
            tokenizer, file_path
        )

    return results


# ------------------------------------------------------------
# Print and save statistics
# ------------------------------------------------------------

def save_and_print_counts(results, output_path="token_counts.json"):
    total_tokens = sum(
        stats["tokens"] for stats in results.values()
    )

    print("\n" + "=" * 75)
    print(
        f"{'Source':<20}{'Documents':>15}"
        f"{'Characters':>18}{'Tokens':>18}"
    )
    print("=" * 75)

    for source, stats in results.items():
        print(
            f"{source:<20}"
            f"{stats['documents']:>15,}"
            f"{stats['characters']:>18,}"
            f"{stats['tokens']:>18,}"
        )

    print("=" * 75)
    print(f"{'TOTAL':<20}{'':>15}{'':>18}{total_tokens:>18,}")

    with open(output_path, "w") as f:
        json.dump(
            {
                "sources": results,
                "total_tokens": total_tokens,
            },
            f,
            indent=2,
        )

    print(f"\nStatistics saved to {output_path}")


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------
tokenizer._build_merge_ranks()
if COUNT_CORPUS_TOKENS:
    results = count_all_sources(tokenizer)
    save_and_print_counts(results)
else:
    print("Skipped full-corpus token counting (COUNT_CORPUS_TOKENS=False).")


Counting supreme_court...


train_supreme_court.json: 0it [00:00, ?it/s]


Counting high_courts...


train_high_courts.json: 0it [00:00, ?it/s]


Counting acts...


train_acts.json: 0it [00:00, ?it/s]


Counting tribunals...


train_tribunals.json: 0it [00:00, ?it/s]


Source                    Documents        Characters            Tokens
supreme_court                55,528     1,239,650,413       253,539,648
high_courts               2,536,738     9,838,813,027     2,262,942,303
acts                         18,822       718,326,398       144,788,040
tribunals                   869,132     6,051,658,029     1,739,512,050
TOTAL                                                     4,400,782,041

Statistics saved to token_counts.json


In [5]:
def run_tokenizer_checks(tokenizer):
    print("Vocabulary size:", len(tokenizer.itos))
    assert len(tokenizer.itos) == len(tokenizer.stoi)
    assert len(tokenizer.itos) <= tokenizer.vocab_size

    # Symbol serialization round trips
    symbols = [
        (97,),
        (97, 98),
        (97, 98, EOW),
        (EOW,),
        (240, 159, 152, 128, EOW),
    ]

    for symbol in symbols:
        encoded = tokenizer.symbol_to_string(symbol)
        decoded = tokenizer.string_to_symbol(encoded)
        assert decoded == symbol, (symbol, encoded, decoded)

    # Text round trips: your current tokenizer normalizes whitespace.
    tests = [
        "The Supreme Court held that the appeal is allowed.",
        "Section 302 of the Indian Penal Code.",
        "Article 14 and Article 21.",
        "A.B.C. v. State of X",
        "न्यायालय",
        "AnUnseenWord123",
    ]

    for text in tests:
        ids = tokenizer.encode(text)
        decoded = tokenizer.decode(ids)

        # Compare against the whitespace-normalized text.
        expected = " ".join(text.split())
        assert decoded == expected, (text, decoded, expected)

    # Save/load consistency
    tokenizer.save("tokenizer_check.json")
    loaded = ByteLevelBPE.load("tokenizer_check.json")

    for text in tests:
        assert tokenizer.encode(text) == loaded.encode(text)

    print("All checks passed.")


run_tokenizer_checks(tokenizer)

Vocabulary size: 16000
Saved: tokenizer_check.json
All checks passed.


## Mixing Strategy

In [9]:

import random
from pathlib import Path
from tqdm.auto import tqdm


# ============================================================
# CORPUS MIXING CONFIGURATION
# Change only this dictionary to tune your data strategy.
# ============================================================

MIXING_STRATEGY = {
    "supreme_court": 1.00,
    "high_courts":    0.30,
    "acts":           1.00,
    "tribunals":      0.50,
}

RANDOM_SEED = 42


def mixed_legal_corpus(
    train_files,
    mixing_strategy=MIXING_STRATEGY,
    seed=RANDOM_SEED,
    show_progress=True,
):
    """
    Stream a mixed legal corpus using document-level sampling.

    Parameters
    ----------
    train_files : dict
        Maps source names to their input file paths.
    mixing_strategy : dict
        Sampling fraction for each source, from 0.0 to 1.0.
    seed : int
        Random seed for reproducibility.
    show_progress : bool
        Whether to show progress bars while reading sources.

    Yields
    ------
    str
        Selected documents in a randomized interleaving
        of the source datasets.

    Notes
    -----
    - Sampling is performed independently per document.
    - The selected token proportions are approximate.
    - Documents are not all loaded into memory.
    """

    rng = random.Random(seed)

    # Validate source names and sampling fractions.
    for source, fraction in mixing_strategy.items():
        if source not in train_files:
            raise ValueError(f"Unknown source: {source}")
        if not 0.0 <= fraction <= 1.0:
            raise ValueError(
                f"Invalid sampling fraction for {source}: {fraction}"
            )

    # Create a document stream for each source.
    source_streams = {}

    for source, fraction in mixing_strategy.items():
        if fraction == 0:
            continue

        file_path = train_files[source]

        def selected_documents(
            file_path=file_path,
            fraction=fraction,
            source=source,
        ):
            documents = iter_texts(file_path)

            if show_progress:
                documents = tqdm(
                    documents,
                    desc=f"Sampling {source}",
                )

            for text in documents:
                if not isinstance(text, str) or not text.strip():
                    continue

                # Keep or skip each document independently.
                if rng.random() < fraction:
                    yield text

        source_streams[source] = iter(selected_documents())

    # Randomly interleave the active source streams.
    active_sources = list(source_streams.keys())

    while active_sources:
        source = rng.choice(active_sources)

        try:
            yield next(source_streams[source])
        except StopIteration:
            active_sources.remove(source)

In [10]:
import json

def save_mixed_corpus(
    output_path="mixed_legal_corpus.jsonl",
    train_files=TRAIN_FILES,
    mixing_strategy=MIXING_STRATEGY,
    seed=42,
):
    total_docs = 0
    total_chars = 0

    with open(output_path, "w", encoding="utf-8") as f:
        for text in mixed_legal_corpus(
            train_files=train_files,
            mixing_strategy=mixing_strategy,
            seed=seed,
        ):
            f.write(json.dumps(
                {"text": text},
                ensure_ascii=False
            ) + "\n")

            total_docs += 1
            total_chars += len(text)

    print(f"Documents saved: {total_docs:,}")
    print(f"Characters saved: {total_chars:,}")
    print(f"Output file: {output_path}")

    return {
        "documents": total_docs,
        "characters": total_chars,
        "output_path": output_path,
    }


WORKING_TOKEN_CACHE = os.path.join(OUTPUT_DIR, "tokenized_data.npz")
if os.path.exists(TOKENIZED_CACHE_INPUT) or os.path.exists(WORKING_TOKEN_CACHE):
    print("Tokenized cache found; mixed corpus not rebuilt.")
else:
    mixing_stats = save_mixed_corpus()

Sampling supreme_court: 0it [00:00, ?it/s]

Sampling acts: 0it [00:00, ?it/s]

Sampling high_courts: 0it [00:00, ?it/s]

Sampling tribunals: 0it [00:00, ?it/s]

Documents saved: 1,271,094
Characters saved: 7,954,993,612
Output file: mixed_legal_corpus.jsonl


In [13]:
# #load it


# import json

# CORPUS_PATH = "/kaggle/input/datasets/richierich6/corpus/combined_corpus.jsonl"

# # Inspect a few documents
# with open(CORPUS_PATH, "r", encoding="utf-8") as f:
#     for i, line in enumerate(f):
#         doc = json.loads(line)
#         text = doc["text"]

#         # Tokenize using your custom ByteLevelBPE tokenizer
#         token_ids = tokenizer.encode(text)

#         print(f"Document {i+1}")
#         print("Characters:", len(text))
#         print("Tokens:", len(token_ids))
#         print("Sample text:", text[:200])
#         print("First 20 token IDs:", token_ids[:20])
#         print("-" * 60)

#         if i == 2:
#             break
# print("Vocabulary size:", tokenizer.vocab_size)
# print("Special tokens:", tokenizer.specials)

# print("First 10 stoi entries:", list(tokenizer.stoi.items())[:10])

# print("Last 10 itos entries:")
# for idx, token in enumerate(
#     tokenizer.itos[-10:],
#     start=len(tokenizer.itos) - 10
# ):
#     print(idx, repr(token))

# # Check whether special token IDs exist
# print("BOS token ID:", getattr(tokenizer, "bos_token_id", None))
# print("EOS token ID:", getattr(tokenizer, "eos_token_id", None))
# print("PAD token ID:", getattr(tokenizer, "pad_token_id", None))

Document 1
Characters: 24257
Tokens: 5290
Sample text: PETITIONER: HIS HOLINESS SRI VISHWOTHAMA THIRTHA SWAMIAR OFSODE MUTT Vs.

RESPONDENT: THE STATE OF MYSORE

DATE OF JUDGMENT: 18/12/1961

ACT: Temple-Large number of pilgrims visiting and worshiping-If
First 20 token IDs: [5968, 7545, 1239, 76, 10711, 2423, 73, 13522, 87, 86, 706, 7869, 76, 91, 8072, 13672, 638, 3844, 77, 86]
------------------------------------------------------------
Document 2
Characters: 7784
Tokens: 1412
Sample text: Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area 
First 20 token IDs: [3574, 3843, 8250, 4510, 5379, 1860, 44, 6107, 290, 9937, 4882, 590, 6019, 9897, 520, 515, 496, 273, 765, 2025]
------------------------------------------------------------
Document 3
Characters: 1056
Tokens: 239
Sample text: Court No. - 5 Case :- WRIT - A No. - 7644 of 1992 Petiti

## 6. Create Train and Validation Tokens Set

In [5]:
import json
import random
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Corpus produced by save_mixed_corpus() above.
train_file_path = "mixed_legal_corpus.jsonl"

# Read the attached cache if present; otherwise build it once in OUTPUT_DIR (writable).
CACHE_PATH = (
    TOKENIZED_CACHE_INPUT
    if Path(TOKENIZED_CACHE_INPUT).exists()
    else os.path.join(OUTPUT_DIR, "tokenized_data.npz")
)

if Path(CACHE_PATH).exists():
    # Reuse previously tokenized data
    cached = np.load(CACHE_PATH)
    train_tokens = cached["train_tokens"].tolist()
    val_tokens = cached["val_tokens"].tolist()

    print("Loaded tokenized data from cache.")

else:
    train_tokens = []
    val_tokens = []
    rng = random.Random(SEED)

    with open(train_file_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Tokenizing corpus"):
            if not line.strip():
                continue

            text = json.loads(line)["text"]
            tokens = tokenizer.encode(text)
            tokens.append(EOS_ID)

            # Split at document level
            if rng.random() < 0.05 and len(val_tokens) < VAL_TOKENS:
                remaining = VAL_TOKENS - len(val_tokens)
                val_tokens.extend(tokens[:remaining])

            elif len(train_tokens) < TRAIN_TOKENS:
                remaining = TRAIN_TOKENS - len(train_tokens)
                train_tokens.extend(tokens[:remaining])

            if (
                len(train_tokens) >= TRAIN_TOKENS
                and len(val_tokens) >= VAL_TOKENS
            ):
                break

    # Save tokens for future reuse
    np.savez(
        CACHE_PATH,
        train_tokens=np.array(train_tokens, dtype=np.int32),
        val_tokens=np.array(val_tokens, dtype=np.int32)
    )

    print("Tokenized data saved to cache.")

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

Loaded tokenized data from cache.
Train tokens: 200,000,000
Validation tokens: 5,000,000


## 7. Create Token Blocks

In [6]:
def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_length]
    return [tokens[i:i + block_size] for i in range(0, total_length, block_size)]

train_blocks = create_blocks(train_tokens, TRAIN_BLOCK_SIZE)
val_blocks = create_blocks(val_tokens, TRAIN_BLOCK_SIZE)

print("Training sequences:", len(train_blocks))
print("Validation sequences:", len(val_blocks))

Training sequences: 97656
Validation sequences: 2441


## 8. Create Datasets

In [7]:
from datasets import Dataset

SOURCE_LENGTH = 1024
TARGET_LENGTH = 1024

def create_seq2seq_examples(blocks):
    input_ids = []
    labels = []

    for block in blocks:
        source = block[:SOURCE_LENGTH]
        target = block[SOURCE_LENGTH:SOURCE_LENGTH + TARGET_LENGTH]

        input_ids.append(source)
        labels.append(target)

    return Dataset.from_dict({
        "input_ids": input_ids,
        "labels": labels
    })

train_dataset = create_seq2seq_examples(train_blocks)
val_dataset = create_seq2seq_examples(val_blocks)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Source length:", len(train_dataset[0]["input_ids"]))
print("Target length:", len(train_dataset[0]["labels"]))


Training examples: 97656
Validation examples: 2441
Source length: 1024
Target length: 1024


## 9. Define The Model

## Architecture 2 - Encoder-Decoder Transformer


In [8]:
PAD_ID = tokenizer.stoi["<pad>"]   # 0
UNK_ID = tokenizer.stoi["<unk>"]   # 1
BOS_ID = tokenizer.stoi["<s>"]     # 2
EOS_ID = tokenizer.stoi["</s>"]    # 3

### Encoder and decoder stacks


In [9]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class EncoderDecoderConfig:
    vocab_size: int
    d_model: int
    d_kv: int
    d_ff: int
    num_layers: int
    num_decoder_layers: int
    num_heads: int
    dropout_rate: float = 0.1
    layer_norm_epsilon: float = 1e-6
    relative_attention_num_buckets: int = 32
    relative_attention_max_distance: int = 128
    feed_forward_proj: str = "relu"
    dense_act_fn: str = "relu"
    tie_word_embeddings: bool = True
    scale_decoder_outputs: bool = True
    initializer_factor: float = 1.0
    pad_token_id: int = 0
    eos_token_id: int = 3
    decoder_start_token_id: int = 0
    max_input_length: int = 1024
    max_target_length: int = 1024
    use_cache: bool = False

    @property
    def inner_dim(self):
        return self.num_heads * self.d_kv

    @property
    def is_gated_act(self):
        return self.feed_forward_proj.startswith("gated-")


class RMSNorm(nn.Module):
    """Scale-only RMS normalization (no mean subtraction, no bias)."""

    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        variance = hidden_states.float().pow(2).mean(dim=-1, keepdim=True)
        normalized = hidden_states * torch.rsqrt(
            variance + self.variance_epsilon
        ).to(dtype=hidden_states.dtype)
        return normalized * self.weight.to(dtype=hidden_states.dtype)


class RelativePositionBias(nn.Module):
    """Bucketed relative-position bias: log-spaced buckets, one learned bias per head."""

    def __init__(
        self,
        num_buckets,
        num_heads,
        max_distance,
        bidirectional,
    ):
        super().__init__()
        if num_buckets < 2:
            raise ValueError("relative_attention_num_buckets must be >= 2")
        if max_distance < 2:
            raise ValueError("relative_attention_max_distance must be >= 2")
        self.num_buckets = num_buckets
        self.num_heads = num_heads
        self.max_distance = max_distance
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(num_buckets, num_heads)

    @staticmethod
    def relative_position_bucket(
        relative_position,
        bidirectional=True,
        num_buckets=32,
        max_distance=128,
    ):
        """Map key_position - query_position to a relative-position bucket."""
        bucket = 0
        if bidirectional:
            num_buckets //= 2
            bucket = bucket + (
                (relative_position > 0).to(torch.long) * num_buckets
            )
            relative_position = torch.abs(relative_position)
        else:
            relative_position = -torch.minimum(
                relative_position, torch.zeros_like(relative_position)
            )

        max_exact = num_buckets // 2
        is_small = relative_position < max_exact
        relative_position_if_large = max_exact + (
            torch.log(
                relative_position.float().clamp(min=1) / max_exact
            )
            / math.log(max_distance / max_exact)
            * (num_buckets - max_exact)
        ).to(torch.long)
        relative_position_if_large = torch.clamp(
            relative_position_if_large, max=num_buckets - 1
        )
        bucket = bucket + torch.where(
            is_small, relative_position, relative_position_if_large
        )
        return bucket

    def forward(self, query_length, key_length, device=None):
        if device is None:
            device = self.embedding.weight.device
        query_positions = torch.arange(
            query_length, dtype=torch.long, device=device
        )[:, None]
        key_positions = torch.arange(
            key_length, dtype=torch.long, device=device
        )[None, :]
        relative_position = key_positions - query_positions
        buckets = self.relative_position_bucket(
            relative_position,
            bidirectional=self.bidirectional,
            num_buckets=self.num_buckets,
            max_distance=self.max_distance,
        )
        values = self.embedding(buckets)
        return values.permute(2, 0, 1).unsqueeze(0)


class MultiHeadAttention(nn.Module):
    """Multi-head attention with bias-free Q/K/V/O projections."""

    def __init__(
        self,
        config: EncoderDecoderConfig,
        causal=False,
        relative_bias_bidirectional=None,
    ):
        super().__init__()
        self.d_model = config.d_model
        self.d_kv = config.d_kv
        self.num_heads = config.num_heads
        self.inner_dim = config.inner_dim
        self.causal = causal

        self.q = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.k = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.v = nn.Linear(self.d_model, self.inner_dim, bias=False)
        self.o = nn.Linear(self.inner_dim, self.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.relative_attention_bias = None
        if relative_bias_bidirectional is not None:
            self.relative_attention_bias = RelativePositionBias(
                num_buckets=config.relative_attention_num_buckets,
                num_heads=config.num_heads,
                max_distance=config.relative_attention_max_distance,
                bidirectional=relative_bias_bidirectional,
            )

    def _split_heads(self, projected):
        batch_size, seq_len, _ = projected.shape
        return projected.view(
            batch_size, seq_len, self.num_heads, self.d_kv
        ).transpose(1, 2)

    def forward(
        self,
        hidden_states,
        key_value_states=None,
        attention_mask=None,
        position_bias=None,
    ):
        if key_value_states is None:
            key_value_states = hidden_states
        query = self._split_heads(self.q(hidden_states))
        key = self._split_heads(self.k(key_value_states))
        value = self._split_heads(self.v(key_value_states))

        scores = torch.matmul(query, key.transpose(-2, -1))
        query_length, key_length = scores.shape[-2:]

        if position_bias is None and self.relative_attention_bias is not None:
            position_bias = self.relative_attention_bias(
                query_length, key_length, device=scores.device
            )
        if position_bias is not None:
            scores = scores + position_bias.to(dtype=scores.dtype)

        if self.causal:
            if query_length != key_length:
                raise ValueError(
                    "This implementation does not use a KV cache; "
                    "causal self-attention requires equal query/key lengths."
                )
            causal_allowed = torch.ones(
                query_length,
                key_length,
                dtype=torch.bool,
                device=scores.device,
            ).tril()
            scores = scores.masked_fill(
                ~causal_allowed.view(1, 1, query_length, key_length),
                torch.finfo(scores.dtype).min,
            )

        if attention_mask is not None:
            if attention_mask.ndim != 2 or attention_mask.shape[-1] != key_length:
                raise ValueError(
                    "attention_mask must have shape [batch, key_length]"
                )
            key_is_valid = attention_mask.to(
                device=scores.device, dtype=torch.bool
            ).view(scores.shape[0], 1, 1, key_length)
            scores = scores.masked_fill(
                ~key_is_valid, torch.finfo(scores.dtype).min
            )

        probabilities = F.softmax(scores.float(), dim=-1).to(
            dtype=scores.dtype
        )
        probabilities = self.dropout(probabilities)
        attended = torch.matmul(probabilities, value)
        attended = attended.transpose(1, 2).contiguous().view(
            attended.shape[0], query_length, self.inner_dim
        )
        return self.o(attended)


class EncoderSelfAttention(MultiHeadAttention):
    def __init__(self, config, has_relative_attention_bias=False):
        super().__init__(
            config,
            causal=False,
            relative_bias_bidirectional=(
                True if has_relative_attention_bias else None
            ),
        )


class DecoderSelfAttention(MultiHeadAttention):
    def __init__(self, config, has_relative_attention_bias=False):
        super().__init__(
            config,
            causal=True,
            relative_bias_bidirectional=(
                False if has_relative_attention_bias else None
            ),
        )


class CrossAttention(MultiHeadAttention):
    def __init__(self, config):
        super().__init__(config, causal=False)


class FeedForward(nn.Module):
    """Feed-forward layer: dense-act-dense, or gated (two input projections)."""

    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.feed_forward_proj = config.feed_forward_proj
        self.activation_name = config.dense_act_fn
        if config.is_gated_act:
            self.wi_0 = nn.Linear(config.d_model, config.d_ff, bias=False)
            self.wi_1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        else:
            self.wi = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.wo = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _activation(self, x):
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("gelu", "gelu_new"):
            return F.gelu(x, approximate="tanh" if self.activation_name == "gelu_new" else "none")
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported dense_act_fn: {self.activation_name}"
        )

    def forward(self, hidden_states):
        if self.feed_forward_proj.startswith("gated-"):
            hidden_states = self._activation(self.wi_0(hidden_states)) * self.wi_1(
                hidden_states
            )
        else:
            hidden_states = self._activation(self.wi(hidden_states))
        hidden_states = self.dropout(hidden_states)
        return self.wo(hidden_states)


class EncoderBlock(nn.Module):
    def __init__(self, config: EncoderDecoderConfig, layer_idx: int):
        super().__init__()
        self.self_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.self_attention = EncoderSelfAttention(
            config,
            has_relative_attention_bias=(layer_idx == 0),
        )
        self.feed_forward_norm = RMSNorm(
            config.d_model, config.layer_norm_epsilon
        )
        self.feed_forward = FeedForward(config)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self, hidden_states, attention_mask=None, position_bias=None,
        return_debug=False
    ):
        debug = {}
        attention_output = self.self_attention(
            self.self_norm(hidden_states),
            attention_mask=attention_mask,
            position_bias=position_bias,
        )
        hidden_states = hidden_states + self.dropout(attention_output)
        if return_debug:
            debug["self_attention_output"] = hidden_states

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(hidden_states)
        )
        hidden_states = hidden_states + self.dropout(feed_forward_output)
        if return_debug:
            debug["feed_forward_output"] = hidden_states
            return hidden_states, debug
        return hidden_states


class DecoderBlock(nn.Module):
    def __init__(self, config: EncoderDecoderConfig, layer_idx: int):
        super().__init__()
        self.self_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.self_attention = DecoderSelfAttention(
            config,
            has_relative_attention_bias=(layer_idx == 0),
        )
        self.cross_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.cross_attention = CrossAttention(config)
        self.feed_forward_norm = RMSNorm(
            config.d_model, config.layer_norm_epsilon
        )
        self.feed_forward = FeedForward(config)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self,
        hidden_states,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
        position_bias=None,
        return_debug=False,
    ):
        debug = {}
        self_output = self.self_attention(
            self.self_norm(hidden_states),
            attention_mask=decoder_attention_mask,
            position_bias=position_bias,
        )
        hidden_states = hidden_states + self.dropout(self_output)
        if return_debug:
            debug["self_attention_output"] = hidden_states

        cross_output = self.cross_attention(
            self.cross_norm(hidden_states),
            key_value_states=encoder_hidden_states,
            attention_mask=encoder_attention_mask,
        )
        hidden_states = hidden_states + self.dropout(cross_output)
        if return_debug:
            debug["cross_attention_output"] = hidden_states

        feed_forward_output = self.feed_forward(
            self.feed_forward_norm(hidden_states)
        )
        hidden_states = hidden_states + self.dropout(feed_forward_output)
        if return_debug:
            debug["feed_forward_output"] = hidden_states
            return hidden_states, debug
        return hidden_states


class Encoder(nn.Module):
    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList(
            EncoderBlock(config, layer_idx)
            for layer_idx in range(config.num_layers)
        )
        self.final_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _run(self, inputs_embeds, attention_mask=None, debug=False):
        hidden_states = self.dropout(inputs_embeds)
        position_bias = self.blocks[0].self_attention.relative_attention_bias(
            hidden_states.shape[1],
            hidden_states.shape[1],
            device=hidden_states.device,
        )
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            if debug:
                hidden_states, block_debug = block(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_bias=position_bias,
                    return_debug=True,
                )
                states[f"block_{layer_idx}_self_attention"] = (
                    block_debug["self_attention_output"]
                )
                states[f"block_{layer_idx}_feed_forward"] = (
                    block_debug["feed_forward_output"]
                )
            else:
                hidden_states = block(
                    hidden_states,
                    attention_mask=attention_mask,
                    position_bias=position_bias,
                )
        hidden_states = self.dropout(self.final_norm(hidden_states))
        states["final_norm"] = hidden_states
        return (hidden_states, states) if debug else hidden_states

    def forward(self, inputs_embeds, attention_mask=None):
        return self._run(inputs_embeds, attention_mask, debug=False)

    def debug_hidden_states(self, inputs_embeds, attention_mask=None):
        return self._run(inputs_embeds, attention_mask, debug=True)


class Decoder(nn.Module):
    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        self.config = config
        self.blocks = nn.ModuleList(
            DecoderBlock(config, layer_idx)
            for layer_idx in range(config.num_decoder_layers)
        )
        self.final_norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _run(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
        debug=False,
    ):
        hidden_states = self.dropout(inputs_embeds)
        position_bias = self.blocks[0].self_attention.relative_attention_bias(
            hidden_states.shape[1],
            hidden_states.shape[1],
            device=hidden_states.device,
        )
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            if debug:
                hidden_states, block_debug = block(
                    hidden_states,
                    encoder_hidden_states,
                    encoder_attention_mask=encoder_attention_mask,
                    decoder_attention_mask=decoder_attention_mask,
                    position_bias=position_bias,
                    return_debug=True,
                )
                for name, value in block_debug.items():
                    states[f"block_{layer_idx}_{name}"] = value
            else:
                hidden_states = block(
                    hidden_states,
                    encoder_hidden_states,
                    encoder_attention_mask=encoder_attention_mask,
                    decoder_attention_mask=decoder_attention_mask,
                    position_bias=position_bias,
                )
        hidden_states = self.dropout(self.final_norm(hidden_states))
        states["final_norm"] = hidden_states
        return (hidden_states, states) if debug else hidden_states

    def forward(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
    ):
        return self._run(
            inputs_embeds,
            encoder_hidden_states,
            encoder_attention_mask,
            decoder_attention_mask,
            debug=False,
        )

    def debug_hidden_states(
        self,
        inputs_embeds,
        encoder_hidden_states,
        encoder_attention_mask=None,
        decoder_attention_mask=None,
    ):
        return self._run(
            inputs_embeds,
            encoder_hidden_states,
            encoder_attention_mask,
            decoder_attention_mask,
            debug=True,
        )


class EncoderDecoderTransformer(nn.Module):
    """Encoder-decoder Transformer, implemented with raw PyTorch layers."""

    def __init__(self, config: EncoderDecoderConfig):
        super().__init__()
        if config.d_model <= 0 or config.d_kv <= 0:
            raise ValueError("d_model and d_kv must be positive")
        if config.num_heads <= 0 or config.num_layers <= 0:
            raise ValueError("num_heads and num_layers must be positive")
        if config.num_decoder_layers <= 0:
            raise ValueError("num_decoder_layers must be positive")
        self.config = config
        self.shared_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.encoder = Encoder(config)
        self.decoder = Decoder(config)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self._initialize_weights()
        if config.tie_word_embeddings:
            # One embedding matrix is shared by the encoder, decoder and LM head.
            self.lm_head.weight = self.shared_embedding.weight

    def _initialize_weights(self):
        factor = self.config.initializer_factor
        with torch.no_grad():
            self.shared_embedding.weight.normal_(mean=0.0, std=factor)
            for module in self.modules():
                if isinstance(module, RMSNorm):
                    module.weight.fill_(1.0)
                elif isinstance(module, RelativePositionBias):
                    module.embedding.weight.normal_(mean=0.0, std=factor)
                elif isinstance(module, MultiHeadAttention):
                    module.q.weight.normal_(
                        mean=0.0,
                        std=factor
                        * (self.config.d_model * self.config.d_kv) ** -0.5,
                    )
                    for projection in (module.k, module.v):
                        projection.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    module.o.weight.normal_(
                        mean=0.0,
                        std=factor * self.config.inner_dim ** -0.5,
                    )
                elif isinstance(module, FeedForward):
                    if self.config.is_gated_act:
                        module.wi_0.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                        module.wi_1.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    else:
                        module.wi.weight.normal_(
                            mean=0.0,
                            std=factor * self.config.d_model ** -0.5,
                        )
                    module.wo.weight.normal_(
                        mean=0.0,
                        std=factor * self.config.d_ff ** -0.5,
                    )

    def _validate_input_ids(self, input_ids, max_length, name):
        if input_ids.ndim != 2:
            raise ValueError(f"{name} must have shape [batch, sequence]")
        if input_ids.shape[1] == 0:
            raise ValueError(f"{name} must not be empty")
        if input_ids.shape[1] > max_length:
            raise ValueError(
                f"{name} length {input_ids.shape[1]} exceeds {max_length}"
            )

    def encode(self, input_ids, attention_mask=None):
        """Return final encoder states with shape [B, L, d_model]."""
        self._validate_input_ids(
            input_ids, self.config.max_input_length, "input_ids"
        )
        if attention_mask is not None and attention_mask.shape != input_ids.shape:
            raise ValueError("attention_mask must match input_ids shape")
        return self.encoder(
            self.shared_embedding(input_ids),
            attention_mask=attention_mask,
        )

    def decode(
        self,
        decoder_input_ids,
        encoder_hidden_states,
        attention_mask=None,
        decoder_attention_mask=None,
    ):
        self._validate_input_ids(
            decoder_input_ids,
            self.config.max_target_length,
            "decoder_input_ids",
        )
        if decoder_input_ids.shape[0] != encoder_hidden_states.shape[0]:
            raise ValueError(
                "decoder and encoder batch dimensions must match"
            )
        return self.decoder(
            self.shared_embedding(decoder_input_ids),
            encoder_hidden_states,
            encoder_attention_mask=attention_mask,
            decoder_attention_mask=decoder_attention_mask,
        )

    def shift_right(self, labels):
        if labels.ndim != 2:
            raise ValueError("labels must have shape [batch, target_length]")
        if labels.shape[1] == 0:
            raise ValueError("labels must not be empty")
        shifted = labels.new_full(labels.shape, self.config.pad_token_id)
        shifted[:, 0] = self.config.decoder_start_token_id
        shifted[:, 1:] = labels[:, :-1]
        shifted.masked_fill_(shifted == -100, self.config.pad_token_id)
        return shifted

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        encoder_hidden_states=None,
    ):
        if encoder_hidden_states is None:
            if input_ids is None:
                raise ValueError(
                    "input_ids are required when encoder states are not supplied"
                )
            encoder_hidden_states = self.encode(input_ids, attention_mask)
        elif input_ids is not None:
            self._validate_input_ids(
                input_ids, self.config.max_input_length, "input_ids"
            )

        if decoder_input_ids is None:
            if labels is None:
                raise ValueError(
                    "Pass decoder_input_ids or labels to run the decoder"
                )
            decoder_input_ids = self.shift_right(labels)
        if decoder_input_ids.shape[1] > self.config.max_target_length:
            raise ValueError("decoder_input_ids exceed max_target_length")

        decoder_hidden_states = self.decode(
            decoder_input_ids,
            encoder_hidden_states,
            attention_mask=attention_mask,
            decoder_attention_mask=decoder_attention_mask,
        )
        if (
            self.config.tie_word_embeddings
            and self.config.scale_decoder_outputs
        ):
            decoder_hidden_states = decoder_hidden_states * (
                self.config.d_model ** -0.5
            )
        logits = self.lm_head(decoder_hidden_states)
        loss = None
        if labels is not None:
            if labels.shape != logits.shape[:2]:
                raise ValueError(
                    "labels must match the decoder logits sequence dimensions"
                )
            loss_labels = labels.masked_fill(
                labels == self.config.pad_token_id, -100
            )
            loss = F.cross_entropy(
                logits.float().reshape(-1, self.config.vocab_size),
                loss_labels.reshape(-1),
                ignore_index=-100,
            )
        return {
            "loss": loss,
            "logits": logits,
            "encoder_hidden_states": encoder_hidden_states,
        }


config = EncoderDecoderConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=512,
    d_kv=64,
    d_ff=2048,
    num_layers=6,
    num_decoder_layers=6,
    num_heads=8,
    dropout_rate=0.1,
    layer_norm_epsilon=1e-6,
    relative_attention_num_buckets=32,
    relative_attention_max_distance=128,
    feed_forward_proj="relu",
    dense_act_fn="relu",
    tie_word_embeddings=True,
    scale_decoder_outputs=True,
    initializer_factor=1.0,
    pad_token_id=PAD_ID,
    eos_token_id=EOS_ID,
    decoder_start_token_id=PAD_ID,
    max_input_length=SOURCE_LENGTH,
    max_target_length=TARGET_LENGTH,
)
model = EncoderDecoderTransformer(config)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"Attention inner dimension: {config.num_heads * config.d_kv}")
print(f"Maximum input length: {config.max_input_length}")


Model parameters: 52,249,088
Model size: 52.25M
Attention inner dimension: 512
Maximum input length: 1024


## Checks

### Checkpoint save / load utilities


In [ ]:
# Checkpoint utilities: weights are stored as a plain state_dict plus the model config.
import json
from dataclasses import asdict
from pathlib import Path

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def save_scratch_checkpoint(scratch_model, output_dir, overwrite=False):
    output_dir = Path(output_dir)
    if output_dir.exists() and any(output_dir.iterdir()) and not overwrite:
        raise FileExistsError(
            f"Refusing to overwrite nonempty scratch checkpoint: {output_dir}"
        )
    output_dir.mkdir(parents=True, exist_ok=True)
    state = {
        key: value.detach().cpu().contiguous()
        for key, value in scratch_model.state_dict().items()
    }
    torch.save(state, output_dir / "pytorch_model.bin")
    (output_dir / "config.json").write_text(
        json.dumps(asdict(scratch_model.config), indent=2),
        encoding="utf-8",
    )
    print("Saved checkpoint:", output_dir.resolve())
    return output_dir


def load_scratch_checkpoint(scratch_model, checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    config_path = checkpoint_dir / "config.json"
    if config_path.is_file():
        saved_config = json.loads(config_path.read_text(encoding="utf-8"))
        expected_config = asdict(scratch_model.config)
        mismatched = {
            key: (saved_config.get(key), value)
            for key, value in expected_config.items()
            if saved_config.get(key) != value
        }
        if mismatched:
            raise ValueError(f"Checkpoint config mismatch: {mismatched}")

    safe_path = checkpoint_dir / "model.safetensors"
    bin_path = checkpoint_dir / "pytorch_model.bin"
    if safe_path.is_file():
        from safetensors.torch import load_file
        state = load_file(str(safe_path), device="cpu")
    elif bin_path.is_file():
        try:
            state = torch.load(
                bin_path, map_location="cpu", weights_only=True
            )
        except TypeError:
            state = torch.load(bin_path, map_location="cpu")
    else:
        raise FileNotFoundError(
            f"No model weights found in {checkpoint_dir}"
        )
    result = scratch_model.load_state_dict(state, strict=True)
    print(
        "Checkpoint load:",
        f"missing={result.missing_keys}, unexpected={result.unexpected_keys}",
    )
    return scratch_model


In [ ]:
# Encoder-decoder sanity checks: architecture, masks, loss, gradients, save/load.
import tempfile

MODEL_NAME = "EncoderDecoder-scratch"
BATCH_SIZE = 4
TEST_BATCH_SIZE = 2

if "train_tokens" in globals() and "val_tokens" in globals():
    assert len(train_tokens) > 0, "Training tokens are empty."
    assert len(val_tokens) > 0, "Validation tokens are empty."
    assert min(train_tokens) >= 0 and max(train_tokens) < tokenizer.vocab_size
    assert min(val_tokens) >= 0 and max(val_tokens) < tokenizer.vocab_size
    print("Dataset/tokenizer checks: PASS")

if "train_blocks" in globals() and "val_blocks" in globals():
    assert TRAIN_BLOCK_SIZE == SOURCE_LENGTH + TARGET_LENGTH
    assert SOURCE_LENGTH <= config.max_input_length
    assert TARGET_LENGTH <= config.max_target_length
    assert len(train_blocks) > 0 and len(val_blocks) > 0
    assert all(len(block) == TRAIN_BLOCK_SIZE for block in train_blocks)
    assert all(len(block) == TRAIN_BLOCK_SIZE for block in val_blocks)
    assert len(train_dataset) > 0 and len(val_dataset) > 0
    assert len(train_dataset[0]["input_ids"]) == SOURCE_LENGTH
    assert len(train_dataset[0]["labels"]) == TARGET_LENGTH
    print("Block and seq2seq dataset checks: PASS")

assert config.vocab_size == tokenizer.vocab_size == len(tokenizer.itos)
assert config.d_model == 512
assert config.d_kv == 64
assert config.d_ff == 2048
assert config.num_layers == 6
assert config.num_decoder_layers == 6
assert config.num_heads == 8
assert config.inner_dim == config.num_heads * config.d_kv
assert config.dropout_rate == 0.1
assert config.layer_norm_epsilon == 1e-6
assert config.relative_attention_num_buckets == 32
assert config.relative_attention_max_distance == 128
assert config.feed_forward_proj == "relu"
assert config.tie_word_embeddings is True
assert config.pad_token_id == PAD_ID == 0
assert config.eos_token_id == EOS_ID == 3
assert config.decoder_start_token_id == PAD_ID
assert model.lm_head.weight is model.shared_embedding.weight

parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
inner_dim = config.inner_dim
expected_parameter_count = (
    config.vocab_size * config.d_model
    + config.num_layers
    * (
        4 * config.d_model * inner_dim
        + 2 * config.d_model * config.d_ff
        + 2 * config.d_model
    )
    + config.relative_attention_num_buckets * config.num_heads
    + config.d_model
    + config.num_decoder_layers
    * (
        8 * config.d_model * inner_dim
        + 2 * config.d_model * config.d_ff
        + 3 * config.d_model
    )
    + config.relative_attention_num_buckets * config.num_heads
    + config.d_model
)
print("Parameter count:", f"{parameter_count:,}")
print("Trainable parameter count:", f"{trainable_parameter_count:,}")
assert parameter_count == expected_parameter_count
assert 50_000_000 <= parameter_count <= 300_000_000

norm = RMSNorm(config.d_model, config.layer_norm_epsilon)
norm_input = torch.linspace(
    -1.0, 1.0, steps=config.d_model, dtype=torch.float32
).unsqueeze(0)
norm_expected = norm_input * torch.rsqrt(
    norm_input.pow(2).mean(dim=-1, keepdim=True)
    + config.layer_norm_epsilon
) * norm.weight
assert torch.allclose(norm(norm_input), norm_expected, atol=1e-7, rtol=1e-7)
assert not hasattr(norm, "bias")
print("RMSNorm scale-only behavior: PASS")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
input_ids = torch.tensor(
    [[PAD_ID, 317, 508, 41, 1297, EOS_ID], [BOS_ID, 19, 22, 311, 7, EOS_ID]],
    dtype=torch.long,
    device=device,
)
attention_mask = torch.tensor(
    [[0, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]],
    dtype=torch.long,
    device=device,
)
labels = torch.tensor(
    [[41, 1297, 22, EOS_ID, -100], [19, 22, 311, 7, EOS_ID]],
    dtype=torch.long,
    device=device,
)
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)
assert outputs["encoder_hidden_states"].shape == (2, 6, config.d_model)
assert outputs["logits"].shape == (2, 5, config.vocab_size)
assert torch.isfinite(outputs["encoder_hidden_states"]).all()
assert torch.isfinite(outputs["logits"]).all()
assert torch.isfinite(outputs["loss"])
assert model.shift_right(
    torch.tensor([[41, 42, -100, 43]], device=device)
).tolist() == [[config.decoder_start_token_id, 41, 42, PAD_ID]]
print("Forward dimensions, shift-right, and finite values: PASS")

# Encoder self-attention must see later source positions.
source_a = torch.tensor([[11, 12, 13, 14]], dtype=torch.long, device=device)
source_b = torch.tensor([[11, 12, 13, 17]], dtype=torch.long, device=device)
with torch.inference_mode():
    encoder_a = model.encode(source_a)
    encoder_b = model.encode(source_b)
assert not torch.allclose(
    encoder_a[:, 0], encoder_b[:, 0], atol=1e-7, rtol=1e-7
)
print("Bidirectional encoder attention: PASS")

# Decoder self-attention must not let future decoder tokens affect earlier logits.
decoder_a = torch.tensor(
    [[PAD_ID, 21, 22, 23, 24]], dtype=torch.long, device=device
)
decoder_b = torch.tensor(
    [[PAD_ID, 21, 22, 23, 97]], dtype=torch.long, device=device
)
fixed_encoder = model.encode(input_ids[:1], attention_mask[:1])
with torch.inference_mode():
    decoder_logits_a = model(
        encoder_hidden_states=fixed_encoder,
        attention_mask=attention_mask[:1],
        decoder_input_ids=decoder_a,
    )["logits"]
    decoder_logits_b = model(
        encoder_hidden_states=fixed_encoder,
        attention_mask=attention_mask[:1],
        decoder_input_ids=decoder_b,
    )["logits"]
assert torch.allclose(
    decoder_logits_a[:, :4],
    decoder_logits_b[:, :4],
    atol=1e-6,
    rtol=1e-6,
)
print("Causal decoder self-attention: PASS")

# A masked source key must not affect the encoder states or cross-attention.
masked_a = torch.tensor(
    [[PAD_ID, 10, 11, 12]], dtype=torch.long, device=device
)
masked_b = torch.tensor(
    [[777, 10, 11, 12]], dtype=torch.long, device=device
)
masked_source = torch.tensor([[0, 1, 1, 1]], dtype=torch.long, device=device)
with torch.inference_mode():
    masked_hidden_a = model.encode(masked_a, masked_source)
    masked_hidden_b = model.encode(masked_b, masked_source)
assert torch.allclose(
    masked_hidden_a[:, 1:], masked_hidden_b[:, 1:], atol=1e-6, rtol=1e-6
)
with torch.inference_mode():
    masked_logits_a = model(
        encoder_hidden_states=masked_hidden_a,
        attention_mask=masked_source,
        decoder_input_ids=decoder_a,
    )["logits"]
    masked_logits_b = model(
        encoder_hidden_states=masked_hidden_b,
        attention_mask=masked_source,
        decoder_input_ids=decoder_a,
    )["logits"]
assert torch.allclose(
    masked_logits_a, masked_logits_b, atol=1e-6, rtol=1e-6
)
print("Encoder padding and cross-attention masks: PASS")

model.train()
model.zero_grad(set_to_none=True)
train_outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)
assert torch.isfinite(train_outputs["loss"])
train_outputs["loss"].backward()
parameters_with_gradients = sum(
    parameter.grad is not None
    for parameter in model.parameters()
    if parameter.requires_grad
)
assert parameters_with_gradients > 0
assert model.encoder.blocks[0].self_attention.q.weight.grad is not None
assert model.decoder.blocks[0].cross_attention.k.weight.grad is not None
print("Finite seq2seq loss and backward gradients: PASS")
model.zero_grad(set_to_none=True)
model.eval()

with tempfile.TemporaryDirectory(prefix="encdec_check_") as temp_dir:
    save_scratch_checkpoint(model, temp_dir)
    reloaded_model = EncoderDecoderTransformer(config).to(device)
    load_scratch_checkpoint(reloaded_model, temp_dir)
    reloaded_model.eval()
    with torch.inference_mode():
        reloaded_logits = reloaded_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )["logits"]
        original_logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )["logits"]
    assert torch.equal(original_logits, reloaded_logits)
    assert reloaded_model.lm_head.weight is reloaded_model.shared_embedding.weight
    del reloaded_model
print("Checkpoint save/load: PASS")

print("=" * 60)
print("GPU / TRAINING WORKLOAD CHECK")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_memory = torch.cuda.get_device_properties(0).total_memory
    print(f"Total GPU memory: {total_memory / 1024**3:.2f} GB")
else:
    print("CUDA is unavailable; training will run on CPU.")
if "train_dataset" in globals():
    steps_per_epoch = math.ceil(len(train_dataset) / BATCH_SIZE)
    print("Architecture:", MODEL_NAME)
    print("Training examples:", len(train_dataset))
    print("Validation examples:", len(val_dataset))
    print("Batch size used for estimate:", BATCH_SIZE)
    print("Epochs:", EPOCHS)
    print("Estimated optimizer steps:", steps_per_epoch * EPOCHS)
print("Encoder-decoder sanity checks passed.")


In [12]:
def data_collator(features):
    input_ids = torch.tensor(
        [f["input_ids"] for f in features],
        dtype=torch.long
    )

    labels = torch.tensor(
        [f["labels"] for f in features],
        dtype=torch.long
    )

    attention_mask = (input_ids != PAD_ID).long()

    # Ignore padding positions in the loss, if padding is present.
    labels[labels == PAD_ID] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

## 10. Training Arguments

In [ ]:
import torch

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/cs781_phase1/encdec_checkpoints",

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,

    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=500,
    lr_scheduler_type="cosine",

    fp16=torch.cuda.is_available(),

    eval_strategy="steps",
    eval_steps=1000,
    logging_steps=100,

    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    seed=SEED,
    report_to="none",

)

## 11. Training

In [16]:
# benchmark encoder-decoder scratch model

import time
import math
import torch
from torch.utils.data import DataLoader

model.train()

# Keep a copy of the initial weights: the benchmark below takes real optimizer steps.
_benchmark_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.1
)

BATCH_SIZE = 4
GRAD_ACCUM = 2
NUM_BENCHMARK_STEPS = 10

bench_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

bench_iter = iter(bench_loader)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

optimizer.zero_grad(set_to_none=True)

start_time = time.time()

for step in range(NUM_BENCHMARK_STEPS):

    for micro_step in range(GRAD_ACCUM):

        try:
            batch = next(bench_iter)
        except StopIteration:
            bench_iter = iter(bench_loader)
            batch = next(bench_iter)

        batch = {
            k: v.to(device)
            for k, v in batch.items()
            if torch.is_tensor(v)
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available()
        ):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch.get("attention_mask"),
                labels=batch["labels"]
            )

            loss = outputs["loss"] / GRAD_ACCUM

        loss.backward()

    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start_time

avg_step_time = elapsed / NUM_BENCHMARK_STEPS

steps_per_epoch = math.ceil(
    len(train_dataset) /
    (BATCH_SIZE * GRAD_ACCUM)
)

total_optimizer_steps = steps_per_epoch * EPOCHS

estimated_hours = (
    avg_step_time *
    total_optimizer_steps /
    3600
)

print(f"Benchmark time: {elapsed:.2f} seconds")
print(
    f"Average optimizer step: "
    f"{avg_step_time:.3f} seconds"
)
print(
    f"Estimated training time: "
    f"{estimated_hours:.2f} hours"
)

if torch.cuda.is_available():
    peak_memory = torch.cuda.max_memory_allocated()

    print(
        f"Peak GPU memory: "
        f"{peak_memory / 1024**3:.2f} GB"
    )

print("Benchmark completed.")

# Restore the pre-benchmark weights so training starts from the same initialization.
model.load_state_dict(_benchmark_state)
del _benchmark_state, optimizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Benchmark time: 9.39 seconds
Average optimizer step: 0.939 seconds
Estimated training time: 9.56 hours
Peak GPU memory: 7.50 GB
Benchmark completed.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

start_time = time.time()

trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

training_time = time.time() - start_time

tokens_per_epoch = len(train_dataset) * TRAIN_BLOCK_SIZE
total_tokens = tokens_per_epoch * EPOCHS

tokens_per_second = total_tokens / training_time

print(f"Training time: {training_time / 3600:.2f} hours")
print(f"Training speed: {tokens_per_second:,.0f} tokens/sec")

In [ ]:
# Evaluate and plot the model trained above.
model = model.to(device)
print("Evaluation model:", type(model).__name__)
logs = trainer.state.log_history


## 12. Perplexity

In [20]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)
print(f"Validation loss: {eval_loss:.6f}")
print(f"Validation perplexity: {perplexity:.4f}")

KeyboardInterrupt: 

## 13. Plot Training and Validation Loss

In [ ]:
import json
import matplotlib.pyplot as plt

if not logs:
    logs = trainer.state.log_history

train_data = [
    (log["step"], log["loss"])
    for log in logs
    if "loss" in log and "step" in log
]

eval_data = [
    (log["step"], log["eval_loss"])
    for log in logs
    if "eval_loss" in log and "step" in log
]

print("Training points:", len(train_data))
print("Validation points:", len(eval_data))
print("First training points:", train_data[:3])
print("First validation points:", eval_data[:3])

plt.figure(figsize=(12, 6))

if train_data:
    x_train, y_train = zip(*train_data)
    plt.plot(x_train, y_train, label="Training Loss", linewidth=1.5)

if eval_data:
    x_eval, y_eval = zip(*eval_data)
    plt.plot(x_eval, y_eval, label="Validation Loss",
             marker="o", markersize=3, linewidth=1.5)

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Encoder-Decoder Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

## 14. Save Final Model to Kaggle Output

In [ ]:
save_scratch_checkpoint(
    model,
    MODEL_OUTPUT_DIR,
    overwrite=True,
)

print("Final model saved to:")
print(MODEL_OUTPUT_DIR)


## 15. Push Model to Hugging Face Hub

In [ ]:
# model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
generation_model = EncoderDecoderTransformer(config)

# Load the final checkpoint saved in section 14.
load_scratch_checkpoint(generation_model, MODEL_OUTPUT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device)
generation_model.eval()

print("Checkpoint:", Path(MODEL_OUTPUT_DIR).resolve())
print("Device:", device)
print(
    f"Parameters: "
    f"{sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M"
)


## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
def _completed_generated_words(token_ids, tokenizer):
    completed = 0
    for token_id in token_ids:
        token = tokenizer.itos[int(token_id)]
        symbol = tokenizer.token_to_symbol.get(token)
        if symbol and symbol[-1] == EOW:
            completed += 1
    return completed


def _banned_ngram_tokens(token_ids, ngram_size):
    if ngram_size <= 0 or len(token_ids) + 1 < ngram_size:
        return set()
    if ngram_size == 1:
        return set(token_ids)
    prefix = tuple(token_ids[-(ngram_size - 1):])
    banned = set()
    for index in range(len(token_ids) - ngram_size + 1):
        ngram = token_ids[index:index + ngram_size]
        if tuple(ngram[:-1]) == prefix:
            banned.add(ngram[-1])
    return banned


def _sample_top_p(logits, temperature, top_p):
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")
    if not 0.0 < top_p <= 1.0:
        raise ValueError("top_p must be in the interval (0, 1]")
    logits = logits.float() / temperature
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probabilities = torch.softmax(sorted_logits, dim=-1)
    cumulative_probabilities = torch.cumsum(sorted_probabilities, dim=-1)
    remove = cumulative_probabilities > top_p
    remove[1:] = remove[:-1].clone()
    remove[0] = False
    logits[sorted_indices[remove]] = -torch.inf
    probabilities = torch.softmax(logits, dim=-1)
    return torch.multinomial(probabilities, num_samples=1).item()


def generate_ids_for_prompt(
    prompt,
    model,
    tokenizer,
    device,
    target_words=128,
    max_new_tokens=MAX_GENERATION_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
):
    if target_words <= 0:
        raise ValueError("target_words must be positive")
    if max_new_tokens <= 0 or max_new_tokens > model.config.max_target_length:
        raise ValueError(
            "max_new_tokens must be positive and no larger than max_target_length"
        )

    source_ids = tokenizer.encode(prompt)[:model.config.max_input_length]
    if not source_ids:
        source_ids = [BOS_ID]
    input_ids = torch.tensor(
        [source_ids], dtype=torch.long, device=device
    )
    attention_mask = torch.ones_like(input_ids)

    # Encode the source once. The same encoder states are reused at every
    # decoder step; only the decoder prefix grows.
    model.eval()
    with torch.inference_mode():
        encoder_hidden_states = model.encode(input_ids, attention_mask)
        decoder_ids = [model.config.decoder_start_token_id]

        for _ in range(max_new_tokens):
            decoder_input_ids = torch.tensor(
                [decoder_ids], dtype=torch.long, device=device
            )
            logits = model(
                attention_mask=attention_mask,
                encoder_hidden_states=encoder_hidden_states,
                decoder_input_ids=decoder_input_ids,
            )["logits"][0, -1].clone()

            if repetition_penalty != 1.0:
                if repetition_penalty <= 0:
                    raise ValueError("repetition_penalty must be positive")
                for token_id in set(decoder_ids):
                    score = logits[token_id]
                    logits[token_id] = (
                        score * repetition_penalty
                        if score < 0
                        else score / repetition_penalty
                    )

            banned = _banned_ngram_tokens(
                decoder_ids, no_repeat_ngram_size
            )
            if banned:
                logits[list(banned)] = -torch.inf

            # Keep EOS from ending a prediction before the requested word
            # quota; generation stops once 128 complete BPE words are made.
            if _completed_generated_words(decoder_ids[1:], tokenizer) < target_words:
                logits[model.config.eos_token_id] = -torch.inf
            logits[model.config.pad_token_id] = -torch.inf
            logits[BOS_ID] = -torch.inf
            logits[UNK_ID] = -torch.inf

            next_token = _sample_top_p(
                logits,
                temperature=temperature,
                top_p=top_p,
            )
            if next_token == model.config.eos_token_id:
                break
            decoder_ids.append(next_token)
            if (
                _completed_generated_words(decoder_ids[1:], tokenizer)
                >= target_words
            ):
                break

    generated_ids = decoder_ids[1:]
    generated_text = tokenizer.decode(generated_ids)
    if len(generated_text.split()) != target_words:
        raise RuntimeError(
            f"Generation produced {len(generated_text.split())} complete words; "
            f"expected exactly {target_words}. Increase max_new_tokens or "
            "inspect the tokenizer word-boundary symbols."
        )
    return generated_ids


def generate_batch(
    prompts,
    model,
    tokenizer,
    device,
    target_words=128,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
):
    predictions = []
    for prompt in prompts:
        generated_ids = generate_ids_for_prompt(
            prompt=prompt,
            model=model,
            tokenizer=tokenizer,
            device=device,
            target_words=target_words,
            max_new_tokens=MAX_GENERATION_TOKENS,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
        )
        text = tokenizer.decode(generated_ids)
        if len(text.split()) != target_words:
            raise RuntimeError(
                f"Expected exactly {target_words} generated words."
            )
        predictions.append(text)
    return predictions


## 20. Generate Predictions for the Test Set

In [ ]:
# Select one actual test prompt.
example_id = list(test_data.keys())[99]
test_prompt = test_data[example_id]["text"]

encoded = tokenizer.encode(test_prompt)[:config.max_input_length]
print("Prompt tokens:", len(encoded))
print("Prompt preview:", repr(tokenizer.decode(encoded)[-300:]))

test_output_ids = generate_ids_for_prompt(
    prompt=test_prompt,
    model=generation_model,
    tokenizer=tokenizer,
    device=device,
    target_words=128,
    max_new_tokens=MAX_GENERATION_TOKENS,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
)
generated_text = tokenizer.decode(test_output_ids)
assert len(generated_text.split()) == 128

print("\nGenerated token IDs:", test_output_ids[:30])
print("\nGenerated token count:", len(test_output_ids))
print("\nGenerated text:")
print(repr(generated_text))
print("\nGenerated words:", len(generated_text.split()))


In [ ]:
predictions = {}
GENERATION_BATCH_SIZE = 32

items = list(test_data.items())
total_examples = len(items)
print(f"Total test examples: {total_examples}")

for start in range(0, total_examples, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    batch_ids = [str(example_id) for example_id, _ in batch]
    batch_prompts = [record["text"] for _, record in batch]

    batch_predictions = generate_batch(
        prompts=batch_prompts,
        model=generation_model,
        tokenizer=tokenizer,
        device=device,
        target_words=128
    )

    for example_id, prediction in zip(batch_ids, batch_predictions):
        predictions[example_id] = {"text": prediction}

    completed = start + len(batch)
    print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%)")

## 21. Save prediction.json

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("prediction.json saved:")
print(PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)